<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h2 align="center"> Aula 07 - Extração de Informação (Parte 2)</h2>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2>1. Extração de Relacionamentos</h2>
<p>A extração de relacionamentos consiste em identificar a ligação entre diversas entidades nomeadas no texto. Isso envolve mencionar qual é o tipo da ligação entre duas entidades. Considere o exemplo de sentença abaixo.</p>

<p>"Carlos Alberto de Nogueira é o morador mais antigo da Rua Praça da Alegria."</p>

<p>Temos as entidades:</p>

* Carlos Alberto de Nogueira (PESSOA)
* Rua Praça da Alegria (LOCALIDADE)

<p>Essas mesmas entidades estão relacionadas da seguinte forma:</p>

[Carlos Alberto de Nogueira (PERSON); morador mais antigo; Rua Praça da Alegria (LOCALIDADE)]


<p>Um dos mais famosos exemplos de sistema de reconhecimento é o [Never-Ending Language Learning (NELL)](http://rtw.ml.cmu.edu/), projeto desenvolvido pela Universidade Carnigie Mellon, com participação do Google e inclusive de pesquisadores brasileiros financiados pelo CNPq. Esse projeto consiste em extrair relacionamentos de milhões de páginas da internet, criando uma gigantesca base de conhecimento.</p>

<h2>2. Métodos para identificação de relacionamentos</h2>

<p>Os métodos mais comuns para identificar relacionamentos entre entidades são:</p>

* **Padrões codificados manualmente**: Basta criar padrões usando expressões regulares, por exemplo, para identificar que duas entidades se relacionam. Assim como em "X mora em Y" pode ser um padrão para identificar o relacionamento (X, mora_em, Y) entre uma entidade X do tipo PESSOA e uma entidade Y do tipo LOCALIDADE.
* **Métodos bootstraping**: Com poucos dados, procura por ocorrências de duas entidades em que já se conhece o relacionamento (no Google, por exemplo), e usa os modelos encontrados como modelos para o mesmo relacionamento entre outras entidades.
* **Métodos supervisionados**: Com base num corpus anotado com relacionamentos, criar modelos que 1) detecte quando existe o relacionamento entre duas entidades e 2) classifique o tipo de relacionamento entre elas. 

<p>Nesta aula, vamos ver um método supervisionado para classificar o relacionamento entre entidades, usando técnicas que já utilizamos em aulas anteriores.</p>

<p>Para isso, utilizaremos alguns atributos mais comuns para o problema, como:</p>

* Bag of Words/LSA ou word2ve
* Flags indicadores dos tipos das entidades
* Número de palavras entre as duas entidades
* Flag indicando se o texto de uma entidade é composto pelo texto da outra
* POS tags
* etc



<h2>3. Criando um Modelo Supervisionado</h2>
<p> Vamos utilizar o corpus [Figure Eight: Medical Sentence Summary](https://www.kaggle.com/kmader/figure-eight-medical-sentence-summary), que possui diversas sentenças extraídas do PubMed, com entidades anotadas, assim como seus tipos de relacionamento.</p>

In [ ]:
import pandas as pd

df_train = pd.read_csv('../input/figure-eight-medical-sentence-summary/train.csv')
df_test = pd.read_csv('../input/figure-eight-medical-sentence-summary/test.csv')

df_train.head(10)

In [ ]:
df_train['relation'].unique()

<p>Transformamos as sentenças e tipos de relacionamento em matrizes numpy. Também binarizamos os rótulos dos relacionamentos, para utilizarmos no nosso classificador logo mais.</p>

In [ ]:
import numpy as np

x_train = df_train['sentence'].as_matrix()
y_train = df_train['relation'].as_matrix()

from sklearn.preprocessing import label_binarize

y_train = label_binarize(y_train, classes=df_train['relation'].unique())

print(x_train[:10])
print(y_train[:10])

<p>Como não temos os tipos das entidades, mas sabemos que se trata de nomes de medicamentos e doenças na maioria dos casos, não utilizaremos o tipo das entidades como atributos, mas utilizaremos os POS tags de todas as palavras entre as entidades. Vamos criar outras matrizes com esses atributos. </p>

<p>Para os POS Tags, vamos fazer algo parecido ao chunking sugerido em https://courses.cs.washington.edu/courses/cse517/13wi/slides/cse517wi13-RelationExtraction.pdf, mas ao invés de usar chunking, vamos criar 3-grams desses POS tags para simplificar.</p>

In [ ]:
#pega o que está entre as entidades
x_train_sub_list = []

for i, row in df_train.iterrows():
    pos_t1 = row['sentence'].find(row['term1'])
    pos_t2 = row['sentence'].find(row['term2']) 
    
    if pos_t1 < pos_t2:#Trata se o 2 aparecer antes do 1
        len_t1 = len(row['term1'])
        x_train_sub_list.append(row['sentence'][pos_t1+len_t1:pos_t2])
    else:
        len_t2 = len(row['term2'])
        x_train_sub_list.append(row['sentence'][pos_t2+len_t2:pos_t1])
    

x_train_sub = np.array(x_train_sub_list)

print(x_train_sub[:10])

<p>Agora vamos definir duas funções de tokenização: uma para tokenizar bag-of-words e outra para tokenizar os POS tags</p>

In [ ]:
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
from nltk.corpus import wordnet

def my_tokenizer_pos(doc):
    words = word_tokenize(doc)#tokeniza em palavras
    
    pos_tags = pos_tag(words)#passa o POS TAG
    
    return [pos[1] for pos in pos_tags]#retorna os POS TAGs de cada palavra

# testando nossa função:

for x in x_train_sub[:10]:
    print(my_tokenizer_pos(x))

In [ ]:
stopwords_list = stopwords.words('english')

lemmatizer = WordNetLemmatizer()

def my_tokenizer_bow(doc):
    words = word_tokenize(doc)
    
    pos_tags = pos_tag(words)
    
    non_stopwords = [w for w in pos_tags if not w[0].lower() in stopwords_list]
    
    non_punctuation = [w for w in non_stopwords if not w[0] in string.punctuation]
    
    lemmas = []
    for w in non_punctuation:
        if w[1].startswith('J'):
            pos = wordnet.ADJ
        elif w[1].startswith('V'):
            pos = wordnet.VERB
        elif w[1].startswith('N'):
            pos = wordnet.NOUN
        elif w[1].startswith('R'):
            pos = wordnet.ADV
        else:
            pos = wordnet.NOUN
        
        lemmas.append(lemmatizer.lemmatize(w[0], pos))

    return lemmas

<p>Vamos reaproveitar a classe para seleção de atributos usando SVD.</p>

In [ ]:
from sklearn.decomposition import TruncatedSVD

class SVDDimSelect(object):
    def fit(self, X, y=None):        
        try:
            self.svd_transformer = TruncatedSVD(n_components=round(X.shape[1]/2))
            self.svd_transformer.fit(X)
        
            cummulative_variance = 0.0#calcula a variancia acumulada até chegar no 0.5
            k = 0
            for var in sorted(self.svd_transformer.explained_variance_ratio_)[::-1]:
                cummulative_variance += var
                if cummulative_variance >= 0.5:
                    break
                else:
                    k += 1
                
            self.svd_transformer = TruncatedSVD(n_components=k)
        except Exception as ex:
            print(ex)
            
        return self.svd_transformer.fit(X)
    
    def transform(self, X, Y=None):
        return self.svd_transformer.transform(X)
        
    def get_params(self, deep=True):
        return {}

<p>Agora vamos criar nosso Pipeline. Em resumo, vamos usar o TFIDF Vectorizer e o nosso POS Tagger em paralelo, e depois juntar os atributos para redimensionar usando o SVD.</p>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
import scipy

clf = OneVsRestClassifier(LogisticRegression(random_state=0, solver='lbfgs', multi_class='multinomial'))
#junta os dois arrays com transformadas diferentes
# o primeiro 'bow'transforma em tridf
my_pipeline = Pipeline([
                        ('union', FeatureUnion([('bow', TfidfVectorizer(tokenizer=my_tokenizer_bow)),\
                                                ('pos', Pipeline([('pos-vect', CountVectorizer(tokenizer=my_tokenizer_pos)), \
                                                         ('pos-tfidf', TfidfTransformer())]))#tokeniza e tira os n gramas, transforma em tfidf
                                               ])),\
                       ('svd', SVDDimSelect()), \
                       ('clf', clf)])

par = {'clf__estimator__C' : np.logspace(-4, 4, 20)}

hyperpar_selector = RandomizedSearchCV(my_pipeline, par, cv=3, scoring='f1_weighted', n_jobs=1, n_iter=20)

<p>Agora vamos treinar os algoritmos</p>

In [ ]:
print(x_train_sub.shape)
print(y_train.shape)

hyperpar_selector.fit(X=x_train_sub[:500], y=y_train[:500])

In [ ]:
x_test = df_test['sentence'].as_matrix()
y_test = df_test['relation'].as_matrix()

y_test = label_binarize(y_test, classes=df_train['relation'].unique())

x_test_sub_list = []

for i, row in df_test.iterrows():
    pos_t1 = row['sentence'].find(row['term1'])
    len_t1 = len(row['term1'])
    
    pos_t2 = row['sentence'].find(row['term2'])    
    
    x_test_sub_list.append(row['sentence'][pos_t1+len_t1:pos_t2])
    

x_test_sub = np.array(x_test_sub_list)

In [ ]:
y_predicted = hyperpar_selector.predict(x_test)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_predicted, target_names=df_train['relation'].unique()))

<p>
</p>

<p><b>Exercício 7:</b> Treine um modelo de extração de relacionamentos em Português, utilizando o corpus extraído do DBPedia e com relacionamentos entre pares de entidades anotadas.</p>

In [5]:
import os
import pandas as pd
import numpy as np
#print(os.listdir('../input'))

In [6]:
df = pd.read_csv('DBpediaRelations-PT-0.2.txt')

ParserError: Error tokenizing data. C error: Expected 9 fields in line 915, saw 12


In [7]:
dict_corpus = {'sentence':[],'term1':[],'term2':[],'relation':[]}
f = open('DBpediaRelations-PT-0.2.txt')
for x in f:
    if ('SENTENCE' in x):
        dict_corpus['sentence'].append(x.split(' : ')[1].replace('\n',''))
    elif ('ENTITY1' in x):
        dict_corpus['term1'].append(x.split(' : ')[1].replace('\n',''))
    elif ('ENTITY2' in x):
        dict_corpus['term2'].append(x.split(' : ')[1].replace('\n',''))
    elif ('REL TYPE' in x):
        dict_corpus['relation'].append(x.split(' : ')[1].replace('\n',''))
f.close()
df_train = pd.DataFrame.from_dict(dict_corpus)
        



UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 7344: character maps to <undefined>

In [79]:
df_train.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98023 entries, 0 to 98022
Data columns (total 4 columns):
sentence    98023 non-null object
term1       98023 non-null object
term2       98023 non-null object
relation    98023 non-null object
dtypes: object(4)
memory usage: 3.0+ MB


In [80]:
data = np.random.rand(len(df)) < 0.7
df_train = df[data]
df_test = df[~data]


In [81]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 29325 entries, 1 to 98021
Data columns (total 4 columns):
sentence    29325 non-null object
term1       29325 non-null object
term2       29325 non-null object
relation    29325 non-null object
dtypes: object(4)
memory usage: 1.1+ MB


In [82]:
import numpy as np

x_train = df_train['sentence'].as_matrix()
y_train = df_train['relation'].as_matrix()

from sklearn.preprocessing import label_binarize

y_train = label_binarize(y_train, classes=df_train['relation'].unique())

print(x_train[:10])
print(y_train[:10])

['A América Latina localiza-se totalmente no hemisfério ocidental, sendo atravessada pelo Trópico de Câncer, que corta a parte central do México; pelo Equador, que passa pelo Brasil, Colômbia, Equador e toca o norte do Peru; e pelo Trópico de Capricórnio, que atravessa o Brasil, o Paraguai, a Argentina e o Chile.'
 'Albert Einstein nasceu na região alemã de Württemberg, na cidade de Ulm, numa família judaica.'
 'É em Ulm que nasce Albert Einstein, em 14 de março de 1879.'
 'Nascido em Itálica na atual Espanha, ou em Roma, na Itália, Adriano era descendente de colonos romanos domiciliados no Sul da Hispânia e primo de Trajano, tendo sido nomeado por este para uma série de dignidades públicas que o fizeram aparecer como herdeiro presuntivo deste imperador.'
 'Talvez por entender que o império esgotara sua capacidade de expansão, Adriano abandonou a política de conquistas de Trajano, adotando outra nitidamente defensiva, optando pela via diplomática para resolver questões relativas ao rel

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:3: FutureWarning: Method .as_matrix will be removed in a future version. Use .values instead.
  This is separate from the ipykernel package so we can avoid doing imports until
/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:4: FutureWarning: Method .as_matrix will be removed in a future version. Use .values instead.
  after removing the cwd from sys.path.


In [83]:
#pega o que está entre as entidades
x_train_sub_list = []

for i, row in df_train.iterrows():
    pos_t1 = row['sentence'].find(row['term1'])
    pos_t2 = row['sentence'].find(row['term2']) 
    
    if pos_t1 < pos_t2:#Trata se o 2 aparecer antes do 1
        len_t1 = len(row['term1'])
        x_train_sub_list.append(row['sentence'][pos_t1+len_t1:pos_t2])
    else:
        len_t2 = len(row['term2'])
        x_train_sub_list.append(row['sentence'][pos_t2+len_t2:pos_t1])
    

x_train_sub = np.array(x_train_sub_list)

print(x_train_sub[:10])

[' localiza-se totalmente no hemisfério ocidental, sendo atravessada pelo Trópico de Câncer, que corta a parte central do México; pelo Equador, que passa pelo Brasil, Colômbia, Equador e toca o norte do Peru; e pelo Trópico de Capricórnio, que atravessa o Brasil, o Paraguai, a '
 ' nasceu na região alemã de ' ' que nasce ' ', '
 ' abandonou a política de conquistas de '
 ' também retificou os limites de uma outra conquista de ' ' ('
 ' morreu em circunstâncias misteriosas, de uma queda de cavalo durante um passeio, em '
 ' foi transferida em 1776 de Kandahar para '
 ' foi transferida em 1776 de ']


In [114]:
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
from nltk.corpus import wordnet
!python -m spacy download pt_core_news_sm
import pt_core_news_sm

nlp = pt_core_news_sm.load()

def my_tokenizer_pos_pt(doc):
    tokens = nlp(doc)
    pos_tags = []
    for t in tokens:
        pos_tags.append(t.pos_)
    
    return pos_tags

# testando nossa função:

#for x in x_train[:10]:
    #print(my_tokenizer_pos_pt(x))

You are using pip version 19.0.3, however version 19.1 is available.
You should consider upgrading via the 'pip install --upgrade pip' command.
✔ Download and installation successful
You can now load the model via spacy.load('pt_core_news_sm')


In [115]:
import nltk
from nltk import RSLPStemmer
stopwords_list = stopwords.words('portuguese')
stemmer = nltk.stem.RSLPStemmer()

def my_tokenizer_bow_pt(doc):
    words = word_tokenize(doc, language='portuguese')
    
    non_stopwords = [w for w in words if not w[0].lower() in stopwords_list]
    
    non_punctuation = [w for w in non_stopwords if not w[0] in string.punctuation]
    
    stemer = []
    for w in non_punctuation:
        stemer.append(stemmer.stem(w))

    return stemer

In [116]:
from sklearn.decomposition import TruncatedSVD

class SVDDimSelect(object):
    def fit(self, X, y=None):        
        try:
            self.svd_transformer = TruncatedSVD(n_components=round(X.shape[1]/2))
            self.svd_transformer.fit(X)
        
            cummulative_variance = 0.0#calcula a variancia acumulada até chegar no 0.5
            k = 0
            for var in sorted(self.svd_transformer.explained_variance_ratio_)[::-1]:
                cummulative_variance += var
                if cummulative_variance >= 0.5:
                    break
                else:
                    k += 1
                
            self.svd_transformer = TruncatedSVD(n_components=k)
        except Exception as ex:
            print(ex)
            
        return self.svd_transformer.fit(X)
    
    def transform(self, X, Y=None):
        return self.svd_transformer.transform(X)
        
    def get_params(self, deep=True):
        return {}

In [117]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
import scipy

clf = OneVsRestClassifier(LogisticRegression(random_state=0, solver='lbfgs', multi_class='multinomial'))
#junta os dois arrays com transformadas diferentes
# o primeiro 'bow'transforma em tridf
my_pipeline = Pipeline([
                        ('union', FeatureUnion([('bow', TfidfVectorizer(tokenizer=my_tokenizer_bow_pt)),\
                                                ('pos', Pipeline([('pos-vect', CountVectorizer(tokenizer=my_tokenizer_pos_pt)), \
                                                         ('pos-tfidf', TfidfTransformer())]))#tokeniza e tira os n gramas, transforma em tfidf
                                               ])),\
                       ('svd', SVDDimSelect()), \
                       ('clf', clf)])

par = {'clf__estimator__C' : np.logspace(-4, 4, 20)}

hyperpar_selector = RandomizedSearchCV(my_pipeline, par, cv=3, scoring='f1_weighted', n_jobs=1, n_iter=20)

In [118]:
import datetime as dt
import warnings
warnings.filterwarnings("ignore")

print(x_train_sub.shape)
print(y_train.shape)

print(dt.datetime.now().strftime("%d-%m-%Y %H:%M:%S"))

hyperpar_selector.fit(X=x_train_sub, y=y_train)

(68698,)
(68698, 10)


/opt/conda/lib/python3.6/site-packages/sklearn/multiclass.py:76: UserWarning: Label not 8 is present in all training examples.
  str(classes[c]))
/opt/conda/lib/python3.6/site-packages/sklearn/multiclass.py:76: UserWarning: Label not 9 is present in all training examples.
  str(classes[c]))
/opt/conda/lib/python3.6/site-packages/sklearn/metrics/classification.py:1143: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)
/opt/conda/lib/python3.6/site-packages/sklearn/metrics/classification.py:1145: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true samples.
  'recall', 'true', average, warn_for)
/opt/conda/lib/python3.6/site-packages/sklearn/metrics/classification.py:1143: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)
/opt/conda/lib/python3.6/sit

RandomizedSearchCV(cv=3, error_score='raise-deprecating',
          estimator=Pipeline(memory=None,
     steps=[('union', FeatureUnion(n_jobs=None,
       transformer_list=[('bow', TfidfVectorizer(analyzer='word', binary=False, decode_error='strict',
        dtype=<class 'numpy.float64'>, encoding='utf-8', input='content',
        lowercase=True, max_df=1.0, max_features=None, min_df=1,
        ngram_r...tate=0, solver='lbfgs',
          tol=0.0001, verbose=0, warm_start=False),
          n_jobs=None))]),
          fit_params=None, iid='warn', n_iter=20, n_jobs=1,
          param_distributions={'clf__estimator__C': array([1.00000e-04, 2.63665e-04, 6.95193e-04, 1.83298e-03, 4.83293e-03,
       1.27427e-02, 3.35982e-02, 8.85867e-02, 2.33572e-01, 6.15848e-01,
       1.62378e+00, 4.28133e+00, 1.12884e+01, 2.97635e+01, 7.84760e+01,
       2.06914e+02, 5.45559e+02, 1.43845e+03, 3.79269e+03, 1.00000e+04])},
          pre_dispatch='2*n_jobs', random_state=None, refit=True,
          return_tra

In [119]:
x_test = df_test['sentence'].as_matrix()
y_test = df_test['relation'].as_matrix()

y_test = label_binarize(y_test, classes=df_train['relation'].unique())

x_test_sub_list = []

for i, row in df_test.iterrows():
    pos_t1 = row['sentence'].find(row['term1'])
    len_t1 = len(row['term1'])
    
    pos_t2 = row['sentence'].find(row['term2'])    
    
    x_test_sub_list.append(row['sentence'][pos_t1+len_t1:pos_t2])
    

x_test_sub = np.array(x_test_sub_list)

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:1: FutureWarning: Method .as_matrix will be removed in a future version. Use .values instead.
  """Entry point for launching an IPython kernel.
/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:2: FutureWarning: Method .as_matrix will be removed in a future version. Use .values instead.
  


In [120]:
y_predicted = hyperpar_selector.predict(x_test)

In [121]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_predicted, target_names=df_train['relation'].unique()))

print(dt.datetime.now().strftime("%d-%m-%Y %H:%M:%S"))


                    precision    recall  f1-score   support

     locatedInArea       0.60      0.60      0.60     13957
            origin       0.74      0.00      0.00      7872
         successor       0.00      0.00      0.00       160
deathOrBurialPlace       0.57      0.39      0.47      2136
             other       0.19      0.09      0.13      3299
         keyPerson       0.00      0.00      0.00       116
            partOf       0.21      0.06      0.09      1586
      influencedBy       0.00      0.00      0.00        50
           partner       0.00      0.00      0.00        57
            parent       0.00      0.00      0.00        92

         micro avg       0.55      0.33      0.41     29325
         macro avg       0.23      0.11      0.13     29325
      weighted avg       0.56      0.33      0.34     29325
       samples avg       0.33      0.33      0.33     29325



/opt/conda/lib/python3.6/site-packages/sklearn/metrics/classification.py:1143: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)
/opt/conda/lib/python3.6/site-packages/sklearn/metrics/classification.py:1143: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels.
  'precision', 'predicted', average, warn_for)
